# 상담 이력 조회와 안전한 데이터 변경

상담 횟수를 집계하고 수정 요청을 DB에 반영한다. 쿼리 결과가 그럴듯해도 빠진 사용자가 없는지 확인한다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 행이 없는 경우와 값이 0인 경우

두 사용자 중 한 명만 상담했다. 상담 테이블에서 집계를 시작하면 상담하지 않은 사용자는 결과 자체에 없다. 모든 사용자를 보여 줄 화면이라면 사용자 테이블에서 시작한다.

LEFT JOIN 뒤에는 매칭되지 않은 행의 상담 ID가 NULL이다. COUNT(*)와 COUNT(상담 ID)가 무엇을 세는지 직접 확인한다.

아래 노트북은 SQLite 메모리 DB로 이 SQL을 재현한다. PostgreSQL의 배포·동시 쓰기 특성은 수업 서비스에서 따로 시험한다.

## 0건인 사용자도 포함하기

두 명을 등록하고 첫 사용자에게만 상담 두 건을 넣는다.

In [1]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE person(id TEXT PRIMARY KEY);
CREATE TABLE turn(id INTEGER PRIMARY KEY, person_id TEXT);
INSERT INTO person VALUES ('a'), ('b');
INSERT INTO turn VALUES (1,'a'), (2,'a');
""")
sql = """SELECT p.id, COUNT(t.id) FROM person p
LEFT JOIN turn t ON p.id = t.person_id GROUP BY p.id ORDER BY p.id"""
print(db.execute(sql).fetchall())

[('a', 2), ('b', 0)]


b도 보이고 값은 0이다. COUNT(*)로 바꾸면 조인 결과의 빈 오른쪽 칸도 한 행으로 세어 1이 된다.

### 취소한 변경이 집계에 남았는지 검사하기

b에게 상담 한 건을 추가한 직후 요청이 취소됐다. 트랜잭션을 되돌리고 같은 집계를 다시 실행한다.

In [2]:
db.execute("INSERT INTO turn VALUES (?,?)", (3,"b"))

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>db.rollback()
print(db.execute(sql).fetchall())</code></pre><pre>[(&#x27;a&#x27;, 2), (&#x27;b&#x27;, 0)]
</pre><p>처리 중 변경이 보였던 연결에서도 rollback 이후 b가 다시 0이 된다. 실제 서비스에서는 오류가 난 경로가 커밋을 호출하는지도 점검한다.</p></details>

## 한 번 취소했던 요청이 재전송되면?

업무 요청 ID를 기본 키로 삼아 동일한 확정 요청이 재전송돼도 접수 한 건만 생기게 한다. 이름이나 요청 시간을 키로 쓰면 같은 고객의 서로 다른 요청이 합쳐질 수 있다.

In [4]:
db.execute("CREATE TABLE IF NOT EXISTS receipt(request_id TEXT PRIMARY KEY, owner TEXT)")
def issue(key, owner):
    with db:
        db.execute("INSERT INTO receipt VALUES (?,?) ON CONFLICT(request_id) DO NOTHING", (key,owner))
issue("R1","a")
issue("R1","a")
print(db.execute("SELECT COUNT(*) FROM receipt WHERE request_id=?", ("R1",)).fetchone())

(1,)


기본 키 충돌 처리로 R1 하나만 저장됐다. 같은 키에 다른 사용자 정보가 들어오는 경우까지 올바르게 처리한 것은 아니다.

### 같은 키에 다른 소유자가 들어온 경우 조사

재전송인지 잘못된 요청인지 결정하려면 기존 소유자를 읽어야 한다. R1에 b를 요청했을 때 기존 값과 같은지 확인한다.

In [5]:
attempted_owner = "b"

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>owner = db.execute(&quot;SELECT owner FROM receipt WHERE request_id=?&quot;,(&quot;R1&quot;,)).fetchone()[0]
print(owner, owner == attempted_owner)</code></pre><pre>a False
</pre><p>서버가 무조건 성공으로 답하면 b는 자신의 요청이 처리됐다고 오해한다. 운영용 쓰기 함수에는 이 불일치를 알릴 경로가 필요하다.</p></details>

## 다른 입력에 적용하기

사용자 둘이 같은 대화 제목을 쓰는 데이터를 넣고 사용자 ID 없이 제목만으로 조인했을 때 어떤 행이 섞이는지 설명한다. PostgreSQL 실습에서는 같은 취소 시나리오를 두 연결로 재현한다.

동료에게 집계 SQL과 변경 전후 행을 넘긴다. 정상 등록·중복 등록·취소에서 어떤 쓰기가 남아야 하는지 재현할 수 있어야 한다.